# PCA in Practice

**Companion lesson:** https://ml-viz.vercel.app/courses/pca-dimensionality/03-pca-in-practice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Compressing 8×8 digits

We PCA scikit-learn's digits (64 pixels), choose components by cumulative variance, and reconstruct.

In [ ]:
from sklearn.datasets import load_digits
X = load_digits().data            # (1797, 64)
mu = X.mean(0); Xc = X - mu
C = np.cov(Xc.T)
vals, vecs = np.linalg.eigh(C)
vals, vecs = vals[::-1], vecs[:, ::-1]   # descending

ratio = vals / vals.sum()
cum = np.cumsum(ratio)
print('components for 90% variance:', np.searchsorted(cum, 0.90) + 1)
print('components for 99% variance:', np.searchsorted(cum, 0.99) + 1)

plt.figure(figsize=(6.5, 4))
plt.plot(cum, color='#6366f1'); plt.axhline(0.95, color='#f43f5e', ls=':')
plt.xlabel('number of components'); plt.ylabel('cumulative explained variance'); plt.show()

## Reconstruction at increasing m

In [ ]:
def reconstruct(x, m):
    z = vecs[:, :m].T @ (x - mu)
    return mu + vecs[:, :m] @ z

digit = X[7]
ms = [2, 8, 21, 64]
fig, axes = plt.subplots(1, len(ms) + 1, figsize=(13, 2.8))
axes[0].imshow(digit.reshape(8, 8), cmap='magma'); axes[0].set_title('original')
for ax, m in zip(axes[1:], ms):
    ax.imshow(reconstruct(digit, m).reshape(8, 8), cmap='magma')
    err = ((reconstruct(digit, m) - digit) ** 2).mean()
    ax.set_title(f'm={m}  mse={err:.1f}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
# reconstruction error per sample averages to the sum of DISCARDED eigenvalues:
print('mean recon MSE at m=8 :', round(np.mean([((reconstruct(x, 8) - x)**2).sum() for x in X[:200]]), 1))
print('sum of discarded λ    :', round(vals[8:].sum(), 1))

## 'Eigendigits' — what the components look like

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(13, 2.2))
for i, ax in enumerate(axes):
    ax.imshow(vecs[:, i].reshape(8, 8), cmap='coolwarm'); ax.axis('off'); ax.set_title(f'PC{i+1}')
plt.tight_layout(); plt.show()
# each component is a pixel-space pattern; every digit = mean + weighted sum of these

**Try it:** reconstruction error as anomaly detector — reconstruct random-noise 'images' with m=8 and compare their error to real digits. Then whiten (`z / np.sqrt(vals[:m])`) and check the coordinates' covariance is the identity.